# Data Processing for Football Analytics

This notebook prepares raw FBref player data for future ML models.
It cleans the data, creates position groups, builds position-specific tables, and exports model-ready CSV files.

In [5]:
import pandas as pd
import os

## Step 1: Load Raw Dataset

Here I read the source CSV file with all player stats.
This gives one raw table that I will clean and reshape.

The CSV was created from FBref data (via the `worldfootballR` package).

In [16]:
folder_path = "data scraping"
file_name = "fbref_big5_players_2019_2025_all_stat_types.csv"
path = os.path.join(folder_path, file_name)
# low_memory=False avoids mixed-type chunk inference warnings.
raw_data = pd.read_csv(path, low_memory=False)
print('Loaded file:', path)

Loaded file: data scraping\fbref_big5_players_2019_2025_all_stat_types.csv


## Step 2: Reshape and Build Core Features

This section does the main preprocessing work.
I split and merge stat types, clean positions, create modeling position groups, clean age, standardize column names, and add `league_strength`.

In [19]:
# create dictionary of stat_type -> list of non-empty columns
# for each stat_type we look at rows of that type and record any column
# where at least one value is not NaN.
df = raw_data.copy()

# --- Canonical key detection (supports either uppercase or lowercase source headers) ---
player_col = 'Player' if 'Player' in df.columns else 'player'
season_col = 'Season_End_Year' if 'Season_End_Year' in df.columns else 'season_end_year'

if player_col not in df.columns or season_col not in df.columns:
    raise KeyError(f"Missing key columns. Available columns: {list(df.columns)}")

# --- Clean join keys early ---
df[player_col] = df[player_col].astype(str).str.strip()
df[player_col] = df[player_col].replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
df[season_col] = pd.to_numeric(df[season_col], errors='coerce').astype('Int64')

# keep only valid key rows for reshape logic
df_valid = df.dropna(subset=[player_col, season_col]).copy()

stat_columns = {}
if 'stat_type' in df_valid.columns:
    for st in df_valid['stat_type'].dropna().unique():
        subset = df_valid[df_valid['stat_type'] == st]
        nonempty = [col for col in subset.columns if subset[col].notna().any()]
        if 'stat_type' in nonempty:
            nonempty.remove('stat_type')
        stat_columns[st] = nonempty
else:
    print('no stat_type column available to inspect')

# also check per stat_type if you want to ensure coverage inside each group
columns_all_na_by_type = {}
if 'stat_type' in df_valid.columns:
    for st in df_valid['stat_type'].dropna().unique():
        subset = df_valid[df_valid['stat_type'] == st]
        cols = subset.columns[subset.isna().all()].tolist()
        columns_all_na_by_type[st] = cols

# split the dataframe by stat_type and merge back to one row per player-season
# start with a clean base of unique player/season combinations
merged = df_valid[[player_col, season_col]].drop_duplicates().copy()

for st, cols in stat_columns.items():
    use_cols = [player_col, season_col] + [c for c in cols if c not in (player_col, season_col, 'stat_type')]
    use_cols = list(dict.fromkeys([c for c in use_cols if c in df_valid.columns]))

    subset = df_valid[df_valid['stat_type'] == st][use_cols].copy()
    subset = subset.drop_duplicates([player_col, season_col])

    # LEFT merge keeps only valid base keys and avoids outer-join key pollution
    merged = pd.merge(
        merged,
        subset,
        on=[player_col, season_col],
        how='left',
        suffixes=('', '_dup')
    )

    dup_cols = [c for c in merged.columns if c.endswith('_dup')]
    if dup_cols:
        for c in dup_cols:
            orig = c[:-4]
            if orig in merged.columns:
                merged[orig] = merged[orig].combine_first(merged[c])
        merged = merged.drop(columns=dup_cols)

# Start from full merged dataframe
df = merged.copy()
df = df.dropna(subset=[player_col, season_col]).copy()

# keep downstream column names expected by existing cells
if player_col != 'Player' and 'Player' not in df.columns:
    df = df.rename(columns={player_col: 'Player'})
if season_col != 'Season_End_Year' and 'Season_End_Year' not in df.columns:
    df = df.rename(columns={season_col: 'Season_End_Year'})

# Clean spacing just in case
df["Pos"] = df["Pos"].astype(str).str.replace(" ", "", regex=False)

# Deterministic mapping exactly as discussed
pos_mapping = {
    "DF,MF": "DEF_MF",
    "MF,DF": "DEF_MF",
    "FW,MF": "ATT_MF",
    "MF,FW": "ATT_MF",
    "DF,FW": "DF",
    "FW,DF": "DF",
}

# Update positions (single positions remain unchanged)
df["Pos_updated"] = df["Pos"].replace(pos_mapping)

def modeling_group(pos):
    if pos == "GK":
        return "GK"
    if pos == "DF":
        return "DF"
    if pos == "FW":
        return "FW"
    if pos in ["MF", "ATT_MF", "DEF_MF"]:
        return "MF"
    return None

df["position_group_model"] = df["Pos_updated"].apply(modeling_group)

df['Age'] = df['Age'].astype(str).str[:2].replace('na', None).astype('Int64')

# convert column names to lowercase
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
)

# remove duplicate column labels safely (keeps first occurrence)
df = df.loc[:, ~df.columns.duplicated()]

# Mapping Opta 'Overall' scores from your image to your exact DF league names
league_weights = {
    'Premier League': 91.2,
    'La Liga': 84.5,
    'Serie A': 84.1,
    'Bundesliga': 84.0,
    'Ligue 1': 83.1
}

# Apply the strength proxy as a new feature
df['league_strength'] = df['comp'].map(league_weights)
print("Step 2 output shape:", df.shape)
print("Null players:", int(df['player'].isna().sum()) if 'player' in df.columns else 'player column missing')
print("season_end_year dtype:", df['season_end_year'].dtype if 'season_end_year' in df.columns else 'season_end_year missing')

Step 2 output shape: (15804, 212)
Null players: 0
season_end_year dtype: Int64


## Step 3: Clean Metrics and Create Per-90 Features

This section removes duplicated and noisy columns, creates ratio features, and converts raw counts to per-90 metrics.
The goal is to keep stable signals and make players comparable across playing time.

In [27]:
import pandas as pd
import numpy as np

def process_football_data_final(df):
    processed_df = df.copy()

   # 1. THE DROP LIST: Redundant, Duplicates, and Raw Totals
# These columns are removed because they are either duplicates, 
# raw counts that have a Per 90 version, or represent noisy over/under performance.
    redundant_and_duplicates = [
    'pos_updated', 'born', 'url','pos',
    
    # Playing Time / Status Duplicates
    'mp_playing.time', 'min_playing.time', 'mn_per_mp_playing.time', 
    'min_percent_playing.time', 'mins_per_90_playing.time', 'starts_starts', 
    'mn_per_start_starts', 'compl_starts', 'mn_per_sub_subs', 'unsub_subs',
    'mins_per_90_playing', 'mins_per_90', 
    
    # Goal / Assist Duplicates (Standard vs Expected variants)
    'gls', 'ast', 'g+a', 'g_minus_pk', 'gls_standard', 'pk_standard', 'pkatt_standard',
    'g+a_per', 'g+a_minus_pk_per', 'xg_expected', 'npxg_expected', 'xag_expected',
    'npxg+xag_expected', 'npxg+xag_per', 'xa', 'xa_expected', 'a_minus_xag_expected','sh_standard', 'sot_standard', 'pkatt_penalty', 'pkm_penalty',
    
    # Performance Over/Under Performance (Residuals that add noise to PCA)
    'g_minus_xg_expected', 'np:g_minus_xg_expected', 'a_minus_xa', 'a_minus_xag',
    
    # Passing / Possession Duplicates & Totals
    'cmp_total', 'cmp_short', 'cmp_medium', 'cmp_long', 'prog', 'xag', 'prgp', 
    'prgc_progression', 'prgp_progression', 'prgr_progression', 'totdist_total', 
    'prgdist_total', 'totdist_carries', 'in_corner', 'out_corner', 'str_corner', 
    'cmp_outcomes', 'att_take', 'succ_take', 'tkld_take', 'tkld_percent_take',
    
    # Defensive Duplicates
    'tkl_challenges', 'tkl_percent_challenges', 'lost_challenges', 'blocks_blocks', 
    'tkl+int', 'clr', 'tklw', 'tklw_tackles', 'won_percent_aerial',
    
    # Team Success / Advanced GK Duplicates
    'plus_per__minus_team.success', 'plus_per__minus_90_team.success', 
    'on_minus_off_team.success', 'xgplus_per__minus__team.success..xg.', 
    'xgplus_per__minus_90_team.success..xg.', 'on_minus_off_team.success..xg.',
    'onxga_team.success..xg', # Added variants to ensure cleanup
    'ga', 'w', 'l', 'cs','d', 'pka_penalty', 'pksv_penalty', 'save_percent_penalty', 
    'pka_goals', 'psxg_expected', 'psxg_per_sot_expected', 'psxg+_per__minus__expected',
    'att_launched', 'opp_crosses', '#opa_sweeper', '#opa_per_90_sweeper'
    ]

# 2. LIST FOR PER 90 CONVERSION
# These are the raw counting stats we want to turn into intensity metrics.
    to_per_90 = [
    'pk', 'pkatt', 'crdy', 'crdr', 'fk_standard', 'att_total', 'att_short', 
    'att_medium', 'att_long', 'kp', 'final_third', 'ppa', 'crspa', 'att', 
    'live_pass', 'dead_pass', 'fk_pass', 'tb_pass', 'sw_pass', 'crs_pass', 
    'ti_pass', 'ck_pass', 'off_outcomes', 'blocks_outcomes', 'tkl_tackles', 
    'def_3rd_tackles', 'mid_3rd_tackles', 'att_3rd_tackles', 'att_challenges', 
    'sh_blocks', 'pass_blocks', 'int', 'err', 'touches_touches', 'def_pen_touches', 
    'def_3rd_touches', 'mid_3rd_touches', 'att_3rd_touches', 'att_pen_touches', 
    'carries_carries', 'prgdist_carries', 'prgc_carries', 'final_third_carries', 
    'cpa_carries', 'mis_carries', 'dis_carries', 'rec_receiving', 'prgr_receiving', 
    'ong_team.success', 'onga_team.success','onxg_team.success..xg.', '2crdy', 'fls', 'fld', 'off', 
    'crs', 'pkwon', 'pkcon', 'og', 'recov', 'won_aerial', 'lost_aerial', 
    'sota', 'saves', 'ga_goals', 'fk_goals', 'ck_goals', 'og_goals', 
    'cmp_launched', 'att_passes', 'thr_passes', 'att_goal', 'stp_crosses', 
    'att_(gk)_passes','live_touches'
    ]

    # 3. CALCULATIONS (Ratios & P90)
    # Ratios before dropping raw counts
    processed_df['pk_success_rate'] = (processed_df['pk'] / processed_df['pkatt']).fillna(0)
    processed_df['pass_cmp_pct'] = (processed_df['cmp_total'] / processed_df['att_total']).fillna(0)
    processed_df['tklw_win_rate'] = (processed_df['tklw_tackles'] / processed_df['tkl_tackles']).fillna(0)
    processed_df['aerial_win_pct'] = (processed_df['won_aerial'] / (processed_df['won_aerial'] + processed_df['lost_aerial'])).fillna(0)
    
    # Style Averages
    processed_df['avrg_dist_pass'] = (processed_df['totdist_total'] / processed_df['att_total']).fillna(0)
    processed_df['avrg_prg_dist_pass'] = (processed_df['prgdist_total'] / processed_df['att_total']).fillna(0)
    processed_df['avrg_dist_per_carry'] = (processed_df['totdist_carries'] / processed_df['carries_carries']).fillna(0)
    
    # P90 Conversion
    for col in to_per_90:
        if col in processed_df.columns:
            processed_df[f'{col}_p90'] = (processed_df[col] / processed_df['min_playing']) * 90
            processed_df[f'{col}_p90'] = processed_df[f'{col}_p90'].replace([np.inf, -np.inf], 0).fillna(0)

    # 4. EXECUTE DROP
    # We drop the explicit redundant list + the raw counts that were converted to P90
    final_drop = redundant_and_duplicates + to_per_90 + ['mp_playing', 'starts_playing', 'min_playing', 'subs_subs', 'pk', 'pkatt']
    processed_df = processed_df.drop(columns=[c for c in final_drop if c in processed_df.columns])

      
    return processed_df

# Run on your initial DF
df_final = process_football_data_final(df)
df_final.to_csv("fbref_ready_to_split.csv", index=False)

## Step 4: Split Into Position Tables

Here I split the cleaned dataset into four tables: GK, DF, MF, FW.
Each table keeps shared metadata plus position-relevant metrics.

In [24]:
import pandas as pd

def split_football_data(df):
    """
    Splits the master dataframe into 4 position-specific tables.
    - Filters rows by position_group_model (GK, DF, MF, FW).
    - Keeps base metadata plus available metric columns for each table.
    - Prints a compact sanity summary for missing metric columns.
    """

    base_cols = [
        'player', 'season_end_year', 'squad', 'comp',
        'nation', 'position_group_model', 'age', 'league_strength'
    ]

    gk_metrics = [
        "cmp_percent_total", "cmp_percent_short", "cmp_percent_medium", "cmp_percent_long",
        "ppm_team.success", "plus_per__minus__team.success", "xgplus_per__minus__team.success..xg",
        "xgplus_per__minus_90_team.success..xg", "on_minus_off_team.success..xg", "ga90",
        "save_percent", "cs_percent", "_per_90_expected", "cmp_percent_launched",
        "launch_percent_passes", "avglen_passes", "launch_percent_goal", "avglen_goal",
        "stp_percent_crosses", "avgdist_sweeper", "league_strength", "pass_cmp_pct",
        "avrg_dist_pass", "avrg_prg_dist_pass", "att_total_p90", "att_short_p90",
        "att_medium_p90", "att_long_p90", "att_p90", "live_pass_p90", "dead_pass_p90",
        "err_p90", "touches_touches_p90", "def_pen_touches_p90", "def_3rd_touches_p90",
        "ong_team.success_p90", "onga_team.success_p90", "onxg_team.success..xg._p90",
        "og_p90", "recov_p90", "sota_p90", "saves_p90", "ga_goals_p90", "fk_goals_p90",
        "ck_goals_p90", "og_goals_p90", "cmp_launched_p90", "att_passes_p90",
        "thr_passes_p90", "att_goal_p90", "stp_crosses_p90", "att_(gk)_passes_p90",
        "live_touches_p90"
    ]

    df_metrics = [
        "gls_per", "ast_per", "g_minus_pk_per", "xg_per", "xag_per", "xg+xag_per",
        "npxg_per", "cmp_percent_total", "cmp_percent_short", "cmp_percent_medium",
        "cmp_percent_long", "ppm_team.success", "plus_per__minus__team.success",
        "xgplus_per__minus__team.success..xg", "xgplus_per__minus_90_team.success..xg",
        "on_minus_off_team.success..xg", "league_strength", "pass_cmp_pct",
        "tklw_win_rate", "aerial_win_pct", "avrg_dist_pass", "avrg_prg_dist_pass",
        "avrg_dist_per_carry", "crdy_p90", "crdr_p90", "att_total_p90", "att_short_p90",
        "att_medium_p90", "att_long_p90", "kp_p90", "final_third_p90", "ppa_p90",
        "crspa_p90", "att_p90", "live_pass_p90", "dead_pass_p90", "fk_pass_p90",
        "tb_pass_p90", "sw_pass_p90", "crs_pass_p90", "ti_pass_p90", "ck_pass_p90",
        "off_outcomes_p90", "blocks_outcomes_p90", "tkl_tackles_p90", "def_3rd_tackles_p90",
        "mid_3rd_tackles_p90", "att_3rd_tackles_p90", "att_challenges_p90", "sh_blocks_p90",
        "pass_blocks_p90", "int_p90", "err_p90", "touches_touches_p90", "def_pen_touches_p90",
        "def_3rd_touches_p90", "mid_3rd_touches_p90", "att_3rd_touches_p90", "carries_carries_p90",
        "prgdist_carries_p90", "prgc_carries_p90", "final_third_carries_p90", "cpa_carries_p90",
        "mis_carries_p90", "dis_carries_p90", "rec_receiving_p90", "prgr_receiving_p90",
        "ong_team.success_p90", "onga_team.success_p90", "onxg_team.success..xg._p90",
        "2crdy_p90", "fls_p90", "fld_p90", "off_p90", "crs_p90", "pkwon_p90", "pkcon_p90",
        "og_p90", "recov_p90", "won_aerial_p90", "lost_aerial_p90", "live_touches_p90"
    ]

    mf_metrics = [
        "gls_per", "ast_per", "g_minus_pk_per", "xg_per", "xag_per", "xg+xag_per",
        "npxg_per", "sot_percent_standard", "sh_per_90_standard", "sot_per_90_standard",
        "g_per_sh_standard", "g_per_sot_standard", "dist_standard", "npxg_per_sh_expected",
        "cmp_percent_total", "cmp_percent_short", "cmp_percent_medium", "cmp_percent_long",
        "succ_percent_take", "ppm_team.success", "plus_per__minus__team.success",
        "xgplus_per__minus__team.success..xg", "xgplus_per__minus_90_team.success..xg",
        "on_minus_off_team.success..xg", "league_strength", "pk_success_rate", "pass_cmp_pct",
        "tklw_win_rate", "aerial_win_pct", "avrg_dist_pass", "avrg_prg_dist_pass",
        "avrg_dist_per_carry", "pk_p90", "pkatt_p90", "crdy_p90", "crdr_p90",
        "fk_standard_p90", "att_total_p90", "att_short_p90", "att_medium_p90",
        "att_long_p90", "kp_p90", "final_third_p90", "ppa_p90", "crspa_p90", "att_p90",
        "live_pass_p90", "dead_pass_p90", "fk_pass_p90", "tb_pass_p90", "sw_pass_p90",
        "crs_pass_p90", "ti_pass_p90", "ck_pass_p90", "off_outcomes_p90",
        "blocks_outcomes_p90", "tkl_tackles_p90", "def_3rd_tackles_p90",
        "mid_3rd_tackles_p90", "att_3rd_tackles_p90", "att_challenges_p90", "sh_blocks_p90",
        "pass_blocks_p90", "int_p90", "err_p90", "touches_touches_p90", "def_pen_touches_p90",
        "def_3rd_touches_p90", "mid_3rd_touches_p90", "att_3rd_touches_p90",
        "att_pen_touches_p90", "carries_carries_p90", "prgdist_carries_p90",
        "prgc_carries_p90", "final_third_carries_p90", "cpa_carries_p90", "mis_carries_p90",
        "dis_carries_p90", "rec_receiving_p90", "prgr_receiving_p90", "ong_team.success_p90",
        "onga_team.success_p90", "onxg_team.success..xg._p90", "2crdy_p90", "fls_p90",
        "fld_p90", "off_p90", "crs_p90", "pkwon_p90", "pkcon_p90", "og_p90", "recov_p90",
        "won_aerial_p90", "lost_aerial_p90", "live_touches_p90"
    ]

    fw_metrics = [
        "gls_per", "ast_per", "g_minus_pk_per", "xg_per", "xag_per", "xg+xag_per",
        "npxg_per", "sot_percent_standard", "sh_per_90_standard", "sot_per_90_standard",
        "g_per_sh_standard", "g_per_sot_standard", "dist_standard", "npxg_per_sh_expected",
        "cmp_percent_total", "cmp_percent_short", "cmp_percent_medium", "cmp_percent_long",
        "succ_percent_take", "ppm_team.success", "plus_per__minus__team.success",
        "xgplus_per__minus__team.success..xg", "xgplus_per__minus_90_team.success..xg",
        "on_minus_off_team.success..xg", "league_strength", "pk_success_rate", "pass_cmp_pct",
        "tklw_win_rate", "aerial_win_pct", "avrg_dist_pass", "avrg_prg_dist_pass",
        "avrg_dist_per_carry", "pk_p90", "pkatt_p90", "crdy_p90", "crdr_p90",
        "fk_standard_p90", "att_total_p90", "att_short_p90", "att_medium_p90",
        "att_long_p90", "kp_p90", "final_third_p90", "ppa_p90", "crspa_p90", "att_p90",
        "live_pass_p90", "dead_pass_p90", "fk_pass_p90", "tb_pass_p90", "sw_pass_p90",
        "crs_pass_p90", "ti_pass_p90", "ck_pass_p90", "off_outcomes_p90",
        "blocks_outcomes_p90", "tkl_tackles_p90", "def_3rd_tackles_p90",
        "mid_3rd_tackles_p90", "att_3rd_tackles_p90", "att_challenges_p90",
        "touches_touches_p90", "mid_3rd_touches_p90", "att_3rd_touches_p90",
        "att_pen_touches_p90", "carries_carries_p90", "prgdist_carries_p90",
        "prgc_carries_p90", "final_third_carries_p90", "cpa_carries_p90", "mis_carries_p90",
        "dis_carries_p90", "rec_receiving_p90", "prgr_receiving_p90", "ong_team.success_p90",
        "onga_team.success_p90", "onxg_team.success..xg._p90", "2crdy_p90", "fls_p90",
        "fld_p90", "off_p90", "crs_p90", "pkwon_p90", "pkcon_p90", "og_p90", "recov_p90",
        "won_aerial_p90", "lost_aerial_p90", "live_touches_p90"
    ]

    table_specs = {
        'GK': gk_metrics,
        'DF': df_metrics,
        'MF': mf_metrics,
        'FW': fw_metrics,
    }

    outputs = {}
    for pos_key, metrics in table_specs.items():
        subset = df[df['position_group_model'] == pos_key].copy()
        requested_cols = list(dict.fromkeys(base_cols + metrics))
        existing_cols = [c for c in requested_cols if c in subset.columns]
        missing_cols = [c for c in requested_cols if c not in subset.columns]

        outputs[pos_key] = subset[existing_cols].copy()
        print(f"{pos_key}: rows={len(outputs[pos_key])}, cols={len(existing_cols)}, missing_requested={len(missing_cols)}")

    return outputs['GK'], outputs['DF'], outputs['MF'], outputs['FW']

# Usage Example
df_gk, df_df, df_mf, df_fw = split_football_data(df_final)

GK: rows=2282, cols=60, missing_requested=0
DF: rows=6122, cols=89, missing_requested=0
MF: rows=4559, cols=102, missing_requested=0
FW: rows=2836, cols=96, missing_requested=0


## Step 5: Build Model-Ready Z-Scored Tables

Here I keep metadata columns unchanged, scale numeric features with z-score, and export one file per position.
This creates clean and comparable inputs for future ML models.

In [ ]:
from pathlib import Path
from sklearn.preprocessing import StandardScaler

BASE_COLUMNS = [
    'player', 'season_end_year', 'squad', 'comp',
    'nation', 'position_group_model', 'age', 'league_strength'
]

def make_model_ready_zscore_table(df_table, table_name, output_dir, base_cols):
    """
    Builds a model-ready table with z-scored metric columns only.
    Metadata/base columns are kept unscaled.
    """
    table = df_table.copy()

    # Keep only base columns that exist in this table.
    present_base_cols = [c for c in base_cols if c in table.columns]

    # Modeling features: everything except base columns.
    feature_cols = [c for c in table.columns if c not in present_base_cols]
    X = table[feature_cols].apply(pd.to_numeric, errors='coerce')

    # Sanity cleanup before scaling.
    all_nan_cols = X.columns[X.isna().all()].tolist()
    if all_nan_cols:
        X = X.drop(columns=all_nan_cols)

    X = X.fillna(X.median(numeric_only=True))

    constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
    if constant_cols:
        X = X.drop(columns=constant_cols)

    if X.shape[1] == 0:
        raise ValueError(f"{table_name}: no valid feature columns left after cleanup")

    scaler = StandardScaler()
    Xz = pd.DataFrame(
        scaler.fit_transform(X),
        columns=X.columns,
        index=table.index
    )

    model_ready = pd.concat([table[present_base_cols], Xz], axis=1)

    # Final finite-value check in feature space.
    finite_mask = np.isfinite(model_ready[Xz.columns].to_numpy()).all()
    if not finite_mask:
        raise ValueError(f"{table_name}: non-finite values found after z-score transformation")

    output_path = Path(output_dir) / f"model_ready_{table_name.lower()}_z.csv"
    model_ready.to_csv(output_path, index=False)

    diagnostics = {
        'table': table_name,
        'rows': int(len(table)),
        'raw_feature_cols': int(len(feature_cols)),
        'kept_feature_cols': int(Xz.shape[1]),
        'dropped_all_nan': int(len(all_nan_cols)),
        'dropped_constant': int(len(constant_cols)),
        'output_path': str(output_path),
    }

    return model_ready, diagnostics

def run_model_ready_export(df_gk, df_df, df_mf, df_fw, output_dir='model_ready'):
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    tables = {
        'GK': df_gk,
        'DF': df_df,
        'MF': df_mf,
        'FW': df_fw,
    }

    outputs = {}
    diagnostics = []

    # Global sanity checks on row keys.
    for name, tdf in tables.items():
        if tdf.empty:
            raise ValueError(f"{name}: table is empty. Check position mapping/filtering.")

        if {'player', 'season_end_year'}.issubset(tdf.columns):
            dup_count = int(tdf.duplicated(['player', 'season_end_year']).sum())
            if dup_count > 0:
                print(f"WARNING [{name}] duplicate player-season rows: {dup_count}")

    for name, tdf in tables.items():
        out_df, diag = make_model_ready_zscore_table(
            df_table=tdf,
            table_name=name,
            output_dir=output_dir,
            base_cols=BASE_COLUMNS,
        )
        outputs[name] = out_df
        diagnostics.append(diag)

    diagnostics_df = pd.DataFrame(diagnostics)

    return outputs, diagnostics_df

model_outputs, model_diagnostics = run_model_ready_export(
    df_gk=df_gk,
    df_df=df_df,
    df_mf=df_mf,
    df_fw=df_fw,
    output_dir='model_ready'
)

model_diagnostics

,table,rows,raw_feature_cols,kept_feature_cols,dropped_all_nan,dropped_constant,output_path
0,GK,2282,52,52,0,0,model_ready\model_ready_gk_z.csv
1,DF,6122,81,81,0,0,model_ready\model_ready_df_z.csv
2,MF,4559,94,94,0,0,model_ready\model_ready_mf_z.csv
3,FW,2836,88,88,0,0,model_ready\model_ready_fw_z.csv
